Q3.

Problem Statement : Linear SVM for Customer Churn Risk — Margin, Support Vectors, and Decision Boundary (Binary Classification)

Syllabus mapping: Optimal decision boundary, margins & support vectors, SVM for linear classification
Objective: Train a Linear SVM classifier to predict churn and produce a clear analysis of margin, support vectors, and generalization using correct metrics.

Dataset (link):

Telco Customer Churn (CSV):

https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv  

Problem Statement

A telecom firm wants a churn prediction model that is both accurate and explainable. You are required to build a Linear SVM model and explain how the decision boundary is formed using support vectors. You must show the impact of regularization (C) on margin width and misclassification.

Expected Input

Telco-Customer-Churn.csv
Program arguments (example):
python task1_linear_svm_churn.py --data Telco-Customer-Churn.csv --target Churn --test_size 0.2 --C 0.1 1 10

Expected Output

For each C value:
Confusion Matrix
Accuracy, Precision, Recall, F1
ROC-AUC
Number of support vectors
Summary table comparing C values (at least 3)
Files:
svm_linear_results.csv (C, metrics, #support_vectors)
test_predictions.csv (CustomerID, Actual, Predicted, Score/Probability)
Mandatory Tasks / Deliverables

Preprocess categorical features (encoding) + scaling of numeric features
Use Stratified train/test split with fixed random seed
Show interpretation: how C affects margin and overfitting/underfitting
Short conclusion: which C generalizes best and why

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [9]:
df = pd.read_csv("Telco-Customer-Churn.csv")


In [11]:
df.shape

(7043, 20)

In [12]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [13]:
df.isnull().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [10]:
df.drop("customerID", axis=1, inplace=True)

In [15]:
# Cconverting churn to binary

df["Churn"] = df["Churn"].map({"Yes":1,"No":0})

X = df.drop("Churn", axis=1)
y = df["Churn"]

In [14]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

/tmp/ipykernel_250/617195598.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


In [16]:
#separating numerical and categorial features

num_cols = X.select_dtypes(include=["int64","float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [17]:
# encoding categorical and scaling numerical

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=0)
#stratified

In [19]:
C_values = [0.1, 1, 10] # regularisation param

results = []
all_predictions = []

SVM tries to do two things:

Make the margin as wide as possible

Correctly classify training points

But sometimes these goals conflict.

The parameter C decides which one is more important

In [21]:
# training svm on multiple c vals
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix

for C in C_values:

    model = Pipeline([
        ("preprocess", preprocessor),
        ("svm", SVC(kernel="linear", C=C, probability=True))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    cm = confusion_matrix(y_test, y_pred)

    support_vectors = model.named_steps["svm"].n_support_.sum()

    print("\nC =", C)
    print("Confusion Matrix\n", cm)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)
    print("ROC AUC:", roc)
    print("Support Vectors:", support_vectors)

    results.append({
        "C":C,
        "accuracy":acc,
        "precision":prec,
        "recall":rec,
        "f1":f1,
        "roc_auc":roc,
        "support_vectors":support_vectors
    })

    temp = pd.DataFrame({
        "Actual":y_test,
        "Predicted":y_pred,
        "Score":y_prob
    })

    all_predictions.append(temp)


C = 0.1
Confusion Matrix
 [[925 110]
 [168 206]]
Accuracy: 0.8026969481902059
Precision: 0.6518987341772152
Recall: 0.5508021390374331
F1: 0.5971014492753624
ROC AUC: 0.8437417654808959
Support Vectors: 2610

C = 1
Confusion Matrix
 [[923 112]
 [166 208]]
Accuracy: 0.8026969481902059
Precision: 0.65
Recall: 0.5561497326203209
F1: 0.5994236311239193
ROC AUC: 0.843860600893849
Support Vectors: 2600

C = 10
Confusion Matrix
 [[924 111]
 [166 208]]
Accuracy: 0.8034066713981547
Precision: 0.6520376175548589
Recall: 0.5561497326203209
F1: 0.6002886002886003
ROC AUC: 0.843674597638792
Support Vectors: 2597


# Interpretation

C is the regularization parameter that controls the trade-off between margin width and misclassification in a Support Vector Machine.

C = 0.1 (strong regularization)

Wider margin, allows more misclassification.

Slightly lower recall and F1.

Highest number of support vectors (2610).

C = 1 (moderate regularization)

Balanced margin and misclassification penalty.

Slight improvement in recall and F1 compared to C=0.1.

Slightly fewer support vectors (2600).

C = 10 (weak regularization)

Narrower margin, stronger penalty for misclassification.

Small improvement in F1 score.

Fewest support vectors (2597).



# Overall observation

Accuracy and ROC-AUC remain almost the same across C values.

Increasing C slightly reduces support vectors but does not significantly improve performance.

# Best generalizing model

C = 1, because it balances margin size and classification performance without overfitting.

Basically, when C is small, the model allows more misclassification but produces a wider margin. This improves generalization but may reduce training accuracy. As C increases, the classifier penalizes misclassification more strongly, resulting in a narrower margin and potentially overfitting the training data

C Value	Effect
Small C (0.1)-->wider margin, more misclassification

Medium C (1)-->balanced

Large C (10)-->narrow margin, risk of overfitting

In [22]:
results_df = pd.DataFrame(results)
results_df.to_csv("linear_svm_results_23102B0056.csv", index=False)

In [23]:
pred_df = pd.concat(all_predictions)

pred_df.to_csv("test_predictions.csv", index=False)